Learning how embeddings work

In [1]:
from sentence_transformers import SentenceTransformer

# Downloads the model automatically on first run (~90MB)
model = SentenceTransformer("all-MiniLM-L6-v2")

vector = model.encode("The dog chased the ball")

print(vector)        # array of 384 floats
print(len(vector))   # 384

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[ 3.54517512e-02  4.64252084e-02  2.45348085e-02  2.83623263e-02
  5.78819737e-02 -1.52010223e-04 -5.13042547e-02  2.76042968e-02
  6.70350716e-02  2.94442307e-02  2.70715711e-04  1.80408042e-02
 -1.30341081e-02 -9.94298514e-03  8.95236880e-02 -2.98368074e-02
 -6.53633699e-02 -3.54418084e-02  2.74623558e-02 -9.39350501e-02
 -4.55109179e-02  8.43798369e-03  5.09549910e-03 -8.71836469e-02
 -9.28841978e-02  3.79634984e-02  1.05602175e-01 -2.67517716e-02
 -8.15140232e-02 -1.33344410e-02  1.15699908e-02 -8.24872106e-02
  1.02015724e-02  7.89964348e-02 -5.49068265e-02 -2.59067304e-02
  8.07068683e-03  1.53064933e-02  8.24737474e-02  2.89681684e-02
 -2.57046777e-03 -2.53251242e-03  4.45794798e-02  4.57244366e-02
 -2.37485464e-03 -1.78396106e-02 -1.24762058e-01 -7.74323195e-02
  2.55313255e-02 -2.03364920e-02 -8.68635401e-02 -3.59784141e-02
 -3.20208110e-02  5.44214174e-02  1.29406266e-02  4.92477342e-02
 -5.03790006e-02  9.30886492e-02 -7.04346690e-04 -4.40408848e-02
 -2.70329006e-02 -7.72408

Comparing Similarity between various embeddings

In [2]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "The dog chased the ball",
    "A puppy ran after the toy",       # similar meaning
    "Quarterly revenue declined 10%"   # unrelated
]

embeddings = model.encode(sentences)

# Compare sentence 0 against sentences 1 and 2
sim1 = util.cos_sim(embeddings[0], embeddings[1])
sim2 = util.cos_sim(embeddings[0], embeddings[2])

print(f"Dog/Puppy similarity:   {sim1.item():.3f}")   # ~0.85 (high)
print(f"Dog/Revenue similarity: {sim2.item():.3f}")   # ~0.05 (low)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Dog/Puppy similarity:   0.526
Dog/Revenue similarity: 0.073


Updated ChromaDB Code (fully local, no API keys)
ChromaDB (often just called Chroma) is an open-source vector database designed to store and search embeddings—numerical representations of text, images, or other data used in AI systems.

User query → Convert to embedding → Search ChromaDB → Retrieve relevant data → Send to LLM → Generate answer

In [4]:
import chromadb
from sentence_transformers import SentenceTransformer

# Local embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Local vector database (stored in memory, no server needed)
client = chromadb.Client()
collection = client.create_collection("my_documents")

# Your documents
documents = [
    "Q3 revenue was $2.3M, up 12% year over year",
    "Product A led sales with 40% market share",
    "Customer churn decreased to 5% this quarter",
    "The weather in Paris is lovely in spring"
]

# Embed and store
embeddings = model.encode(documents).tolist()

collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=[f"doc{i}" for i in range(len(documents))]
)

# Query
query = "what were the top products?"
query_embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2
)

for doc in results["documents"][0]:
    print(doc)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Product A led sales with 40% market share
Q3 revenue was $2.3M, up 12% year over year
